# Evaluación final del score de pago tardío (B)

**Proyecto Final Integrador · Machine Learning I · Universidad EAN**
Integrantes: Claudia Naranjo, Yery Pedraza y Nicolás Ruiz.

Calibración, lista diaria, prueba y backtest del modelo final preliminar (regresión logística),
con el diseño de calibración, lista diaria y backtest y la calibración sin pesos de clase.

| Sección | Qué hace | ¿Usa la prueba? |
|---|---|---|
| §1 Calibración | Calibra la logística y la verifica en la validación de R12 | No |
| §2 Ensayo del backtest | Corre el backtest completo en el periodo de validación, para depurarlo | No |
| §3 **Consulta única a la prueba** | Métricas técnicas y backtest del periodo de prueba | **Sí, una sola vez** |

**Confidencialidad:** solo cifras agregadas. Los montos están escalados (aproximados).

## 0. Configuración

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos_crudos").is_dir())
sys.path.insert(0, str(RAIZ / "src"))

from cartera import backtest as bt
from cartera import datos
from cartera import diseno as dis
from cartera import historial as hi
from cartera import muestra as mu
from cartera.config import SEMILLA
from cartera.diseno import CAT, CORTE, FECHA_CORTE, NUM

np.random.seed(SEMILLA)
print("Semilla", SEMILLA, "| Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

Semilla 2026 | Python 3.14.7 | pandas 3.0.6 | numpy 2.5.3 | scikit-learn 1.9.1


In [2]:
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline

saldos, movs = datos.leer_todo()
conj = dis.conjuntos(saldos, movs)
subent, val, ent, prueba = (conj[n] for n in ("subentrenamiento", "validación", "entrenamiento completo", "prueba"))

def logistica(pesos="balanced"):
    """La logística de R13 (pesos="balanced") o el modelo calibrado sin pesos (pesos=None)."""
    return Pipeline([("prep", dis.prep_lineal()),
                     ("est", LogisticRegression(class_weight=pesos, max_iter=2000))])

def calibrada(datos_ent):
    """Modelo de la lista diaria: logística sin class_weight, calibrada por cómo se ajusta."""
    return logistica(pesos=None).fit(datos_ent[NUM + CAT], datos_ent.tardia)

def metricas(y, p, n_boot=1000):
    """AUC-PR, AUC-ROC y Brier, con intervalo de bootstrap del 95 % (remuestreo de facturas)."""
    y, p = np.asarray(y), np.asarray(p)
    rng = np.random.default_rng(SEMILLA)
    fun = {"AUC-PR": average_precision_score, "AUC-ROC": roc_auc_score, "Brier": brier_score_loss}
    boot = {k: [] for k in fun}
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        for k, f in fun.items():
            boot[k].append(f(y[i], p[i]))
    return pd.DataFrame({k: {"valor": f(y, p), "IC 95 % inferior": np.percentile(boot[k], 2.5),
                             "IC 95 % superior": np.percentile(boot[k], 97.5)} for k, f in fun.items()}).T.round(3)

def tabla_calibracion(y, p):
    frac, media = calibration_curve(y, p, n_bins=10, strategy="quantile")
    return pd.DataFrame({"probabilidad media predicha": media, "proporción real de tardías": frac}).round(3)

## 1. Calibración (sin la prueba)

La logística de R13 se entrena con `class_weight = "balanced"`, que sube las probabilidades de todas
las facturas. Para multiplicarlas por el saldo tienen que estar calibradas: si dice 30 %, cerca del
30 % de esas facturas debe atrasarse. El diseño inicial proponía calibrarla con el método *sigmoid*; el ensayo en
validación mostró que calibraba mal y bajaba la AUC-PR. Por eso el modelo de la lista es la
**misma logística sin `class_weight`**: sin pesos, la regresión logística da probabilidades
calibradas por cómo se ajusta.

**Verificación en la validación de R12:** las dos versiones se entrenan con el subentrenamiento y se
miden en la validación (2024-07 a 2025-06).

In [3]:
X_sub, y_sub = subent[NUM + CAT], subent.tardia
X_val, y_val = val[NUM + CAT], val.tardia
p_sin = logistica().fit(X_sub, y_sub).predict_proba(X_val)[:, 1]
cal_val = calibrada(subent)
p_cal = cal_val.predict_proba(X_val)[:, 1]
pd.concat({"con class_weight (R13)": metricas(y_val, p_sin)["valor"],
           "sin class_weight": metricas(y_val, p_cal)["valor"]}, axis=1)

,con class_weight (R13),sin class_weight
AUC-PR,0.724,0.723
AUC-ROC,0.844,0.841
Brier,0.152,0.145


In [4]:
pd.concat({"con class_weight (R13)": tabla_calibracion(y_val, p_sin),
           "sin class_weight": tabla_calibracion(y_val, p_cal)}, axis=1)

con class_weight (R13)                             \
  probabilidad media predicha proporción real de tardías   
0                       0.068                      0.026   
1                       0.119                      0.071   
2                       0.191                      0.079   
3                       0.255                      0.143   
4                       0.315                      0.162   
5                       0.390                      0.211   
6                       0.476                      0.357   
7                       0.597                      0.492   
8                       0.754                      0.714   
9                       0.938                      0.831   

      sin class_weight                             
  probabilidad media predicha proporción real de tardías  
0                       0.034                      0.030  
1                       0.060                      0.068  
2                       0.104                      0.098  
3                       0.140                      0.135  
4                       0.176                      0.158  
5                       0.226                      0.211  
6                       0.286                      0.351  
7                       0.386                      0.483  
8                       0.555                      0.722  
9                       0.848                      0.831

## 2. Ensayo del backtest en el periodo de validación (sin la prueba)

Se corre el backtest completo con el modelo sin pesos de clase, entrenado con el subentrenamiento, en los días hábiles
de 2024-07-01 a 2025-05-30, con los desenlaces que se conocían el 1/07/2025. Sirve para depurar el
cálculo antes de usar la prueba; sus cifras no son el resultado del proyecto.

- **Cartera del día *t*:** facturas abiertas con su saldo reconstruido con los movimientos anteriores
  a *t* (`src/cartera/backtest.py`). Al 18/10/2026 coincide factura por factura con el saldo real.
- **Listas de K clientes:** regla actual, saldo de 1 a 30 días y modelo.
- **Métrica principal:** del dinero en duda (no vencido o vencido hace 30 días o menos) que terminó
  tardío, qué parte estaba en la lista, sumando todos los días.

In [5]:
# Probabilidad de cada factura que puede estar abierta en los días simulados, calculada al emitirla
facturas = saldos[saldos.es_deuda_facturada & (saldos.plazo_dias_observado >= 0) & (saldos.fecha >= "2021-06-01")]
facturas = mu.agregar_log_monto_inicial(facturas, movs)
facturas = facturas.join(hi.calcular(facturas, saldos, movs))
cartera = bt.Cartera(saldos, movs)

def probabilidad(modelo):
    return pd.Series(modelo.predict_proba(facturas[NUM + CAT])[:, 1], index=facturas.id_saldo)

dias_ensayo = pd.bdate_range("2024-07-01", "2025-05-30")
sim_ensayo = bt.simular(cartera, dias_ensayo, probabilidad(cal_val), bt.desenlace_final(saldos, movs, CORTE))
bt.resumir(sim_ensayo)

monto en riesgo capturado (%)  \
k  lista                                                        
10 modelo                                                40.1   
   regla actual                                   25.9   
   saldo de 1 a 30 días                           35.0   
20 modelo                                                57.1   
   regla actual                                   44.3   
   saldo de 1 a 30 días                           50.4   
40 modelo                                                73.0   
   regla actual                                   63.9   
   saldo de 1 a 30 días                           68.5   

                                complementaria: todo el dinero tardío capturado (%)  \
k  lista                                                                              
10 modelo                                                                    35.0     
   regla actual                                                       35.7     
   saldo de 1 a 30 días                                               27.3     
20 modelo                                                                    50.4     
   regla actual                                                       54.5     
   saldo de 1 a 30 días                                               40.5     
40 modelo                                                                    65.0     
   regla actual                                                       75.0     
   saldo de 1 a 30 días                                               56.2     

                                complementaria: dinero ya tardío en la lista (% del saldo de la lista)  
k  lista                                                                                                
10 modelo                                                                    37.3                       
   regla actual                                                       61.9                       
   saldo de 1 a 30 días                                               26.9                       
20 modelo                                                                    38.1                       
   regla actual                                                       56.8                       
   saldo de 1 a 30 días                                               27.7                       
40 modelo                                                                    37.0                       
   regla actual                                                       51.8                       
   saldo de 1 a 30 días                                               28.8

## 3. Consulta única a la prueba

**Esta sección se ejecuta una sola vez.** Se ejecutó el 21/10/2026 a las 22:09; nada se cambió después. El modelo final (logística sin `class_weight`) se
reentrena con todo el entrenamiento (2022-01 a 2025-06). Después:

1. **Métricas técnicas** en las 3.970 facturas de prueba (2025-07 a 2026-09), con intervalos de
   bootstrap y la línea base (la proporción de tardías).
2. **Backtest** en los días hábiles del 1/07/2025 al 18/08/2026, con los desenlaces conocidos al
   18/10/2026.

Lo que salga aquí no se usa para cambiar el modelo, la lista ni la métrica.

In [6]:
modelo_final = calibrada(ent)
X_pr, y_pr = prueba[NUM + CAT], prueba.tardia
p_pr = modelo_final.predict_proba(X_pr)[:, 1]
tecnicas = metricas(y_pr, p_pr)
print(f"Facturas de prueba: {len(prueba)} | tardías: {y_pr.mean() * 100:.1f} % (AUC-PR de la línea base)")
tecnicas

Facturas de prueba: 3970 | tardías: 38.6 % (AUC-PR de la línea base)


,valor,IC 95 % inferior,IC 95 % superior
AUC-PR,0.744,0.721,0.768
AUC-ROC,0.829,0.816,0.842
Brier,0.177,0.170,0.185


In [7]:
tabla_calibracion(y_pr, p_pr)

,probabilidad media predicha,proporción real de tardías
0,0.024,0.038
1,0.055,0.128
2,0.095,0.161
3,0.125,0.156
4,0.168,0.214
5,0.226,0.385
6,0.320,0.516
7,0.432,0.647
8,0.568,0.781
9,0.809,0.836


In [8]:
dias_prueba = pd.bdate_range(CORTE, "2026-08-18")
sim = bt.simular(cartera, dias_prueba, probabilidad(modelo_final), bt.desenlace_final(saldos, movs, FECHA_CORTE))
resultado = bt.resumir(sim)
resultado

monto en riesgo capturado (%)  \
k  lista                                                        
10 modelo                                                32.0   
   regla actual                                   24.0   
   saldo de 1 a 30 días                           26.9   
20 modelo                                                47.4   
   regla actual                                   37.9   
   saldo de 1 a 30 días                           41.4   
40 modelo                                                67.1   
   regla actual                                   57.3   
   saldo de 1 a 30 días                           60.1   

                                complementaria: todo el dinero tardío capturado (%)  \
k  lista                                                                              
10 modelo                                                                    27.4     
   regla actual                                                       30.6     
   saldo de 1 a 30 días                                               20.9     
20 modelo                                                                    43.7     
   regla actual                                                       46.5     
   saldo de 1 a 30 días                                               33.8     
40 modelo                                                                    62.4     
   regla actual                                                       66.2     
   saldo de 1 a 30 días                                               52.9     

                                complementaria: dinero ya tardío en la lista (% del saldo de la lista)  
k  lista                                                                                                
10 modelo                                                                    27.0                       
   regla actual                                                       45.4                       
   saldo de 1 a 30 días                                               18.0                       
20 modelo                                                                    30.4                       
   regla actual                                                       44.7                       
   saldo de 1 a 30 días                                               20.9                       
40 modelo                                                                    30.8                       
   regla actual                                                       41.6                       
   saldo de 1 a 30 días                                               25.2

In [9]:
por_dia = sim[sim.k == 20].assign(pct=lambda d: d.capturado / d.riesgo * 100).pivot(index="dia", columns="lista", values="pct")
unicos = sim.drop_duplicates("dia")
pd.Series({
    "Días hábiles simulados": len(dias_prueba),
    "Dinero en duda sin desenlace conocido o sin etiqueta (% del dinero en duda)":
        round(unicos.duda_desconocido.sum() / unicos.duda_total.sum() * 100, 1),
    "Días en que el modelo captura más que la regla actual (K = 20)":
        f"{int((por_dia['modelo'] > por_dia['regla actual']).sum())} de {len(por_dia)}",
    "Días en que el modelo captura más que la segunda línea base (K = 20)":
        f"{int((por_dia['modelo'] > por_dia['saldo de 1 a 30 días']).sum())} de {len(por_dia)}",
    "Mediana diaria del capturado, modelo (K = 20)": round(por_dia["modelo"].median(), 1),
    "Mediana diaria del capturado, regla actual (K = 20)": round(por_dia["regla actual"].median(), 1),
    "Mediana diaria del capturado, segunda línea base (K = 20)": round(por_dia["saldo de 1 a 30 días"].median(), 1),
}, dtype=object).to_frame("valor")

,valor
Días hábiles simulados,296
Dinero en duda sin desenlace conocido o sin etiqueta (% del dinero en duda),6.6
Días en que el modelo captura más que la regla actual (K = 20),254 de 296
Días en que el modelo captura más que la segunda línea base (K = 20),200 de 296
"Mediana diaria del capturado, modelo (K = 20)",45.8
"Mediana diaria del capturado, regla actual (K = 20)",38.0
"Mediana diaria del capturado, segunda línea base (K = 20)",42.4


### Lectura de los resultados de la prueba

**Métricas técnicas.** AUC-PR 0,744 (IC 95 %: 0,721 a 0,768), frente a 0,386 de la línea base, y
AUC-ROC 0,829. El orden de las facturas se sostiene en un periodo que el modelo no vio.

**Calibración: subestima.** En la prueba hay más tardías (38,6 %) que en el entrenamiento
(≈ 30 %), y la probabilidad queda corta: donde predice 32 %, se atrasa el 52 %. El orden sigue
siendo bueno, pero la probabilidad no se puede leer al pie de la letra. Ese 38,6 % sale del rango
de seguimiento del anteproyecto §7 (31–34 % anual): según ese plan, el modelo habría que
revisarlo y reentrenarlo.

**Backtest (métrica principal), K = 20.** El modelo captura el **47,4 %** del dinero en
duda que terminó tardío, frente al **37,9 %** de la regla actual y el **41,4 %** de la
segunda línea base. Le gana a la regla actual en 254 de 296 días y a la segunda línea base
en 200 de 296. Con K = 10 y K = 40 el orden es el mismo.

**Tabla complementaria, siempre secundaria.** Contando todo el dinero tardío, también el
que ya estaba decidido, la regla actual capta más (46,5 % frente a 43,7 %): el 44,7 % del saldo de
su lista ya era dinero tardío, frente al 30,4 % en la del modelo. Es lo esperado: la regla mira lo
que ya se atrasó y el modelo anticipa lo que se va a atrasar.

**Lo que no se mide aquí:** si llamar a esos clientes acorta el atraso. El backtest compara qué
dinero estaba en cada lista, no el efecto de la gestión; eso se valida con el piloto (anteproyecto
§5, C).